# CatBoost CPU — 20-Seed Average (Full Data)

CPU version of the seed-averaging experiment, using float64 precision.  
CPU single model: OOF 0.95544, LB **0.95357** (new best, +0.00001 over GPU 0.95356).

**Approach**: train 20 models on full training data, average test predictions.  
No early stopping (no eval set) — use fixed iterations based on CPU fold averages.  
Saves each seed's predictions immediately — **resumable if interrupted**.

**Estimated runtime**: ~15-20 min/model × 20 seeds = 5-6 hours

In [1]:
import subprocess
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
FEATURES = [c for c in train.columns if c not in ['heart_disease', 'id']]

X      = train[FEATURES]
y      = train['heart_disease'].values
X_test = test[FEATURES]

# Fixed iterations: CPU fold means were 955-1524, avg ~1100.
# Use 1300 — slightly above average, safe for full-data training.
N_SEEDS    = 20
ITERATIONS = 1300
CKPT_DIR   = Path('submissions/cpu_seedavg_ckpts')
CKPT_DIR.mkdir(exist_ok=True)

BASE_PARAMS = dict(
    iterations    = ITERATIONS,
    learning_rate = 0.05,
    depth         = 6,
    task_type     = 'CPU',
    thread_count  = -1,
    cat_features  = CAT_FEATURES,
    verbose       = 0,
)

print(f'Seeds: {N_SEEDS}  Iterations per model: {ITERATIONS}  task_type: CPU')
print(f'Checkpoints: {CKPT_DIR}')

Seeds: 20  Iterations per model: 1300  task_type: CPU
Checkpoints: submissions/cpu_seedavg_ckpts


In [2]:
fit_times = []

for seed in range(N_SEEDS):
    ckpt = CKPT_DIR / f'seed_{seed:02d}.npy'

    if ckpt.exists():
        print(f'seed={seed:2d}  [loaded from checkpoint]')
        continue

    t0 = time.time()
    m  = CatBoostClassifier(**BASE_PARAMS, random_seed=seed)
    m.fit(X, y)
    preds = m.predict_proba(X_test)[:, 1]
    np.save(ckpt, preds)

    elapsed = time.time() - t0
    fit_times.append(elapsed)

    # Show running average mean/std as a stability check
    completed = sorted(CKPT_DIR.glob('seed_*.npy'))
    running   = np.stack([np.load(f) for f in completed]).mean(axis=0)
    remaining = N_SEEDS - len(completed)
    eta_min   = remaining * np.mean(fit_times) / 60 if fit_times else '?'

    print(f'seed={seed:2d}  {elapsed/60:.1f}min  '
          f'running_mean={running.mean():.4f}  '
          f'({len(completed)}/{N_SEEDS} done  ETA ~{eta_min:.0f}min)')

print('\nAll seeds complete.')

seed= 0  1.2min  running_mean=0.4498  (1/20 done  ETA ~23min)
seed= 1  1.2min  running_mean=0.4498  (2/20 done  ETA ~21min)
seed= 2  1.2min  running_mean=0.4497  (3/20 done  ETA ~20min)
seed= 3  1.2min  running_mean=0.4498  (4/20 done  ETA ~19min)
seed= 4  1.2min  running_mean=0.4498  (5/20 done  ETA ~18min)
seed= 5  1.3min  running_mean=0.4498  (6/20 done  ETA ~17min)
seed= 6  1.2min  running_mean=0.4498  (7/20 done  ETA ~16min)
seed= 7  1.2min  running_mean=0.4498  (8/20 done  ETA ~15min)
seed= 8  1.3min  running_mean=0.4498  (9/20 done  ETA ~13min)
seed= 9  1.6min  running_mean=0.4498  (10/20 done  ETA ~13min)
seed=10  1.2min  running_mean=0.4498  (11/20 done  ETA ~11min)
seed=11  1.2min  running_mean=0.4498  (12/20 done  ETA ~10min)
seed=12  1.3min  running_mean=0.4498  (13/20 done  ETA ~9min)
seed=13  1.2min  running_mean=0.4498  (14/20 done  ETA ~8min)
seed=14  1.2min  running_mean=0.4498  (15/20 done  ETA ~6min)
seed=15  1.2min  running_mean=0.4498  (16/20 done  ETA ~5min)
seed=

In [5]:
# Load all checkpoints and average
ckpts      = sorted(CKPT_DIR.glob('seed_*.npy'))
all_preds  = np.stack([np.load(f) for f in ckpts])
test_preds = all_preds.mean(axis=0)

print(f'Averaged {len(ckpts)} seeds')
print(f'Prediction mean={test_preds.mean():.4f}  std={test_preds.std():.4f}')

# Compare vs GPU seed avg
gpu_avg = np.load('submissions/test_cat_seedavg_fulldata.npy')
corr = np.corrcoef(test_preds, gpu_avg)[0, 1]
diff = np.abs(test_preds - gpu_avg).mean()
print(f'vs GPU seed avg: corr={corr:.6f}  mean_abs_diff={diff:.6f}')

label = f'cat_cpu_seedavg_{len(ckpts)}seeds'
fname = f'submissions/{label}.csv'
sub   = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)
print(f'Saved: {fname}')

desc = f'{label} | cv_auc=NA'
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')

Averaged 20 seeds
Prediction mean=0.4498  std=0.4072
vs GPU seed avg: corr=0.999861  mean_abs_diff=0.003213
Saved: submissions/cat_cpu_seedavg_20seeds.csv
cat_cpu_seedavg_20seeds: submitted
